# Pipeline de Embeddings RDF2Vec (leakage-free)

Genera embeddings semánticos a partir del grafo RDF (`kg_accidentes_bcn.ttl`) mediante
**random walks + Word2Vec** (RDF2Vec) y los serializa en `outputs/` para su uso en
`clustering_analysis.ipynb`.

**Prevención de leakage:** se excluyen del grafo todos los predicados y clases que
codifican directamente la severidad del accidente (`NumMort`, `NumFeritGreu`,
`AccidenteGrave`, `AccidenteMortal`, etc.) antes de generar los walks.

## 1. Imports

In [ ]:
import os
import random
import urllib.parse
import numpy as np
import pandas as pd
import rdflib
from rdflib import Graph, Namespace, RDF
from gensim.models import Word2Vec
import gensim

print("Gensim version:", gensim.__version__)
print("RDFLib version:", rdflib.__version__)

## 2. Cargar el grafo RDF

In [ ]:
KG_PATH = "kg_accidentes_bcn.ttl"
g = Graph()
print(f"Cargando RDF graph desde {KG_PATH}  (~1 min)...")
g.parse(KG_PATH, format="turtle")
print(f"Grafo cargado: {len(g):,} triples")

## 3. Generador de walks leakage-free

Construye una lista de adyacencia no dirigida filtrando:
- **Predicados de fuga:** `NumMort`, `NumFeritLleu`, `NumFeritGreu`.
- **Clases de severidad** en triples `rdf:type`: `AccidenteLeve`, `AccidenteGrave`,
  `AccidenteMortal`, `AccidenteSinVictima`.

Así el vector resultante de cada nodo no codifica la severidad directamente.

In [ ]:
leakage_predicates = {
    "http://webprotege.stanford.edu/RDLF2iVJgjiRFE4TYcqZ6a3",  # NumMort
    "http://webprotege.stanford.edu/R8RNUHdxy4iNBMh6DiKMNHR",  # NumFeritLleu
    "http://webprotege.stanford.edu/R9KzZ5w47b2GzTS1wFlEzB6",  # NumFeritGreu
}
leakage_classes = {
    "http://webprotege.stanford.edu/R7oZUszqj7aEjlsk698JmFM",  # AccidenteLeve
    "http://webprotege.stanford.edu/RYqhWnA3Yc3qHHEFBZcRoJ",  # AccidenteGrave
    "http://webprotege.stanford.edu/R8bzivJwhQYj45eHewQTg08",  # AccidenteMortal
    "http://webprotege.stanford.edu/RCyHtlgkOu2iks7ZQbQPeLw",  # AccidenteSinVictima
}
RDF_TYPE = "http://www.w3.org/1999/02/22-rdf-syntax-ns#type"

class LeakageFreeWalkGenerator:
    def __init__(self, graph):
        self.adj = {}
        for s, p, o in graph:
            s_str, p_str, o_str = str(s), str(p), str(o)
            if p_str in leakage_predicates:
                continue
            if p_str == RDF_TYPE and o_str in leakage_classes:
                continue
            self.adj.setdefault(s_str, []).append((p_str, o_str))
            self.adj.setdefault(o_str, []).append((p_str + "_inv", s_str))

    def random_walk(self, start, depth):
        walk, curr = [start], start
        for _ in range(depth):
            neighbors = self.adj.get(curr, [])
            if not neighbors:
                break
            rel, nxt = random.choice(neighbors)
            walk.extend([rel, nxt])
            curr = nxt
        return walk

    def generate_walks(self, start_nodes, num_walks, depth):
        walks = []
        for node in start_nodes:
            for _ in range(num_walks):
                walks.append(self.random_walk(node, depth))
        return walks

## 4. Generar walks sobre nodos Accidente

In [ ]:
WP = Namespace("http://webprotege.stanford.edu/")
C_ACCIDENTE = WP["RDHGr7pl4B25glJmq2xYYmN"]

accident_nodes = [str(s) for s in g.subjects(RDF.type, C_ACCIDENTE)]
print(f"Nodos Accidente encontrados: {len(accident_nodes):,}")

random.seed(42)
walk_gen = LeakageFreeWalkGenerator(g)
walks = walk_gen.generate_walks(accident_nodes, num_walks=30, depth=4)
print(f"Walks generados: {len(walks):,}")

## 5. Entrenar Word2Vec (RDF2Vec)

In [ ]:
print("Entrenando Word2Vec skip-gram...")
w2v = Word2Vec(
    sentences=walks,
    vector_size=64,
    window=5,
    min_count=1,
    sg=1,
    workers=4,
    epochs=20,
    seed=42,
)
print(f"Modelo entrenado. Vocabulario: {len(w2v.wv):,} términos")

## 6. Extraer embedding por accidente

In [ ]:
embedding_dict = {}
for node in accident_nodes:
    if node in w2v.wv:
        exp_id = urllib.parse.unquote(node.split("/")[-1])
        embedding_dict[exp_id] = w2v.wv[node]

print(f"Embeddings extraídos: {len(embedding_dict):,} accidentes")

## 7. Serializar → `outputs/`

Guarda la matriz de embeddings y el índice de expedientes para que
`clustering_analysis.ipynb` los cargue directamente sin repetir el proceso.

In [ ]:
os.makedirs("outputs", exist_ok=True)

exp_ids    = list(embedding_dict.keys())
emb_matrix = np.stack([embedding_dict[e] for e in exp_ids])

np.save("outputs/embeddings.npy", emb_matrix)
pd.DataFrame({"numero_expedient": exp_ids}).to_csv("outputs/embeddings_index.csv", index=False)

print(f"outputs/embeddings.npy       {emb_matrix.shape}")
print(f"outputs/embeddings_index.csv {len(exp_ids):,} filas")